In [ ]:
import pandas as pd
import numpy as np
import ollama
import re
import json

from src.io import read_tabular
from pathlib import Path

from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Literal

In [2]:
#Loading Data

fp = Path("../../data/test_semantic_extraction.csv")
df = read_tabular(fp)
input = df['sentence']

### Define Pydantic class:
Sentence - Subject - Verb - Object

Desired output: Series of JSON objects with the four vars, no grouping by sentence

In [ ]:
class SVO(BaseModel):
    sentence: str = Field(..., description="The full grammatical sentence you extracted the SVO from")
    subject: str = Field(..., description="The semantic subject related to the verb")
    verb: str = Field(..., description="The verb of the sentence or partial sentence")
    object: str = Field(..., description="The semantic object related to the verb")

svo_schema = SVO.model_json_schema()

{'properties': {'sentence': {'description': 'The full grammatical sentence you extracted the SVO from', 'title': 'Sentence', 'type': 'string'}, 'subject': {'description': 'The semantic subject related to the verb', 'title': 'Subject', 'type': 'string'}, 'verb': {'description': 'The verb of the sentence or partial sentence', 'title': 'Verb', 'type': 'string'}, 'object': {'description': 'The semantic object related to the verb', 'title': 'Object', 'type': 'string'}}, 'required': ['sentence', 'subject', 'verb', 'object'], 'title': 'SVO', 'type': 'object'}


### Prompting

In [7]:
sysprompt_svo_zs = f'''
Your task is to read a sentence and extract semantic actors. For each action in the sentence, extract the following four variables:

- "sentence": The full grammatical sentence you were passed to analyze.
- "subject": The semantic subject associated with the action.
- "verb": The verb that indicates the action, in its infinitive form.
- "object": The semantic object associated with the verb.

Observe the following rules:
- Subject and object should always be understood as semantic, rather than grammatical, categories. If there are indirect reports, passive sentences, etc. it is important that you parse the full sentence and identify the relevant semantic relations.
- All variables should be answered using words already in the text. If the subject or object is referenced using a pronoun, return that pronoun.
- Return the entire description of subject and object, including articles, potential adjectives, etc.

Output rules:
Return a JSON using the following scheme:
{svo_schema}

Return a new entry for each action in the sentence. Do not wrap in a code block. Do not return anything else.'''

### Setup

- Gemma-Cloud OR GPT-OSS 20b
- Temp = 0
- Seed = 42

In [ ]:
modelname = "gemma4:31b-cloud"

ollama.pull(modelname)
client = ollama.Client()

error_ctr=0
success_ctr=0
out = []
json_out = []

for text in input:
    messages = [
            {"role": "system", "content": sysprompt_svo_zs},
            {"role": "user", "content": text}
        ]

    opts = {
        "seed": 42,
        "temperature": 0.0
    }

    response = client.chat(
        model=modelname, 
        messages=messages, 
        options=opts,
        format=svo_schema)

    ctr = ctr+1
    print(f"Processed {ctr} of {len(input)} sentences")

    out.append(response)

    # Validate and save on running basis
    try:
        parsed = json.loads(response["message"]["content"])        
        validated = SVO(**parsed)
        json_out.append(validated.model_dump())
        success_ctr = success_ctr + 1
          
    except (json.JSONDecodeError, TypeError, ValidationError) as e:
        print(f"Error validating final output: {e}")
        error_ctr = error_ctr + 1
        continue
          
    with open(f"svo_{modelname}.json", "w", encoding="utf-8") as f:
        json.dump(json_out, f, indent=2, ensure_ascii=False)
          
    with open(f"svo_{modelname}_raw.txt", "w", encoding="utf-8") as f:
        f.write(f"{out}")
          
    print(f"Processed {success_ctr} texts successfully, {error_ctr} errors encountered.")

In [36]:
output = out[1]
test = output.get("message", {}).get("content", "{}")
test = json.loads(test)
#svo_inst = SVO(**test)
print(output)
print(test)

model='gemma4:31b' created_at='2026-10-07T15:40:04.219240061Z' done=True done_reason='stop' total_duration=457716648 load_duration=None prompt_eval_count=414 prompt_eval_duration=None eval_count=83 eval_duration=None message=Message(role='assistant', content='[\n  {\n    "sentence": "In a roadmap published on Wednesday, the German government set out for the first time an explicit promise of “transitioning away” from fossil fuels.",\n    "subject": "the German government",\n    "verb": "set out",\n    "object": "an explicit promise of “transitioning away” from fossil fuels"\n  }\n]', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
[{'sentence': 'In a roadmap published on Wednesday, the German government set out for the first time an explicit promise of “transitioning away” from fossil fuels.', 'subject': 'the German government', 'verb': 'set out', 'object': 'an explicit promise of “transitioning away” from fossil fuels'}]


In [40]:
val_out = []

for output in out:
    content = output.get("message", {}).get("content")
    try:
        svo_instance = SVO(**content)
        val_out.append(svo_instance.model_dump())
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Error parsing output: {e}")
        continue


TypeError: __main__.SVO() argument after ** must be a mapping, not str